In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os
import zipfile

zip_path = "/content/RAW CNN.zip"
extract_path = "/content/dataset_b"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)

In [ ]:

for root, dirs, files in os.walk("/content/dataset_b/raw"):
    level = root.replace("/content/dataset_b/raw", "").count(os.sep)

    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")

In [ ]:
from pathlib import Path

# Let's search for where the actual class folders are extracted under dataset_b
extract_base = Path("/content/dataset_b")

# Find a directory that contains our target subfolders (like '0_healthy')
candidate_roots = []
for p in extract_base.rglob("0_healthy"):
    if p.is_dir():
        candidate_roots.append(p.parent)

if candidate_roots:
    dataset_root = candidate_roots[0]
else:
    # Fallback to the original path if nothing is found
    dataset_root = Path("/content/dataset_b/agrovision_leaf_dataset - Copy/train")

if not dataset_root.exists():
    raise FileNotFoundError(f"Could not find the dataset folder at: {dataset_root}")

print("Dataset root verified successfully:")
print(dataset_root)
print("\nFound classes:")
for subclass in sorted(dataset_root.iterdir()):
    if subclass.is_dir():
        print(f"- {subclass.name}")

In [ ]:
classes = [
    "0_healthy",
    "1_water_stress",
    "2_heat_stress",
    "3_nutrient_deficiency",
    "4_pollution"
]

for class_name in classes:
    class_path = dataset_root / class_name

    if class_path.exists():
        print(f"✓ {class_name}")
    else:
        print(f"✗ MISSING: {class_name}")

In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png",
    ".bmp", ".webp", ".tif", ".tiff"
}

class_counts = {}

for class_name in classes:
    class_path = dataset_root / class_name

    count = sum(
        1 for file in class_path.rglob("*")
        if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS
    )

    class_counts[class_name] = count

print("Class distribution:\n")

total = 0

for class_name, count in class_counts.items():
    print(f"{class_name:25} {count}")
    total += count

print("-" * 40)
print(f"{'TOTAL':25} {total}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.bar(
    class_counts.keys(),
    class_counts.values()
)

plt.title("Dataset B Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from PIL import Image

records = []

for class_name in classes:

    class_path = dataset_root / class_name

    for file in class_path.rglob("*"):

        if not file.is_file():
            continue

        if file.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        try:
            with Image.open(file) as img:

                img.verify()

            with Image.open(file) as img:

                width, height = img.size
                mode = img.mode
                format_name = img.format

            records.append({
                "file": str(file),
                "class": class_name,
                "width": width,
                "height": height,
                "mode": mode,
                "format": format_name,
                "valid": True,
                "error": ""
            })

        except Exception as e:

            records.append({
                "file": str(file),
                "class": class_name,
                "width": None,
                "height": None,
                "mode": None,
                "format": None,
                "valid": False,
                "error": str(e)
            })

df = pd.DataFrame(records)

print("Total image files:", len(df))
print("Valid images:", df["valid"].sum())
print("Invalid images:", (~df["valid"]).sum())

In [ ]:
REPORT_DIR = Path("/content/dataset_b/reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(
    REPORT_DIR / "dataset_audit.csv",
    index=False
)

print("Saved:")
print(REPORT_DIR / "dataset_audit.csv")

In [ ]:
bad_images = df[df["valid"] == False]

if len(bad_images) == 0:
    print("✓ No corrupted images detected.")
else:
    print("Corrupted/unreadable images:")
    display(bad_images)

In [ ]:
valid_df = df[df["valid"]].copy()

print("Image dimension summary:\n")

print(
    valid_df[
        ["width", "height"]
    ].describe()
)

In [ ]:
dimension_summary = (
    valid_df
    .groupby("class")[["width", "height"]]
    .agg(["min", "max", "mean"])
)

display(dimension_summary)

In [ ]:
print(
    valid_df["mode"].value_counts()
)

In [ ]:
import hashlib
from collections import defaultdict


def calculate_md5(path):

    md5 = hashlib.md5()

    with open(path, "rb") as f:

        for chunk in iter(lambda: f.read(8192), b""):
            md5.update(chunk)

    return md5.hexdigest()


hash_groups = defaultdict(list)

for _, row in valid_df.iterrows():

    path = row["file"]

    try:

        image_hash = calculate_md5(path)
        hash_groups[image_hash].append(path)

    except Exception:
        pass


duplicate_groups = {
    h: paths
    for h, paths in hash_groups.items()
    if len(paths) > 1
}

print("Unique image hashes:", len(hash_groups))
print("Duplicate groups:", len(duplicate_groups))

duplicate_files = sum(
    len(paths) - 1
    for paths in duplicate_groups.values()
)

print(
    "Duplicate files beyond first copy:",
    duplicate_files
)

In [ ]:
for i, (image_hash, paths) in enumerate(
    duplicate_groups.items()
):

    print(f"\nDuplicate group {i + 1}")

    for path in paths:
        print(" ", path)

    if i >= 9:
        break

In [ ]:
!pip -q install ImageHash

In [ ]:
import imagehash
from PIL import Image

phash_groups = defaultdict(list)

for _, row in valid_df.iterrows():

    path = row["file"]

    try:

        with Image.open(path) as img:

            img = img.convert("RGB")
            h = str(imagehash.phash(img))

        phash_groups[h].append(path)

    except Exception:
        pass

near_duplicate_groups = {
    h: paths
    for h, paths in phash_groups.items()
    if len(paths) > 1
}

print(
    "Potential near-duplicate groups:",
    len(near_duplicate_groups)
)

In [ ]:
import shutil

CLEAN_DIR = Path("/content/dataset_b/cleaned")

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Valid files only
valid_files = set(
    valid_df["file"].tolist()
)

copied = 0

for _, row in valid_df.iterrows():

    source = Path(row["file"])
    class_name = row["class"]

    destination_dir = CLEAN_DIR / class_name
    destination_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    destination = destination_dir / source.name

    # Avoid filename collision
    if destination.exists():

        stem = source.stem
        suffix = source.suffix

        counter = 1

        while destination.exists():

            destination = (
                destination_dir /
                f"{stem}_{counter}{suffix}"
            )

            counter += 1

    shutil.copy2(source, destination)

    copied += 1

print("Images copied to cleaned dataset:", copied)
print("Clean dataset:", CLEAN_DIR)

In [ ]:
for class_name in classes:

    class_dir = CLEAN_DIR / class_name

    count = sum(
        1 for file in class_dir.iterdir()
        if file.is_file()
    )

    print(f"{class_name:25} {count}")

In [ ]:
import shutil
from pathlib import Path

from sklearn.model_selection import train_test_split

SPLIT_DIR = Path("/content/dataset_b/split")

for split_name in ["train", "val", "test"]:
    for class_name in classes:
        (SPLIT_DIR / split_name / class_name).mkdir(
            parents=True,
            exist_ok=True
        )

In [ ]:
all_files = []
all_labels = []

for class_name in classes:

    class_dir = CLEAN_DIR / class_name

    for file in class_dir.iterdir():

        if file.is_file():

            all_files.append(str(file))
            all_labels.append(class_name)


train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files,
    all_labels,
    test_size=0.30,
    stratify=all_labels,
    random_state=42
)

val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=42
)

print("Train:", len(train_files))
print("Validation:", len(val_files))
print("Test:", len(test_files))
print("Total:", len(train_files) + len(val_files) + len(test_files))

In [ ]:
def copy_split(files, labels, split_name):

    for file, label in zip(files, labels):

        source = Path(file)

        destination = (
            SPLIT_DIR /
            split_name /
            label /
            source.name
        )

        shutil.copy2(source, destination)


copy_split(
    train_files,
    train_labels,
    "train"
)

copy_split(
    val_files,
    val_labels,
    "val"
)

copy_split(
    test_files,
    test_labels,
    "test"
)

print("Train/validation/test split completed.")

In [ ]:
for split_name in ["train", "val", "test"]:

    print("\n", split_name.upper())

    for class_name in classes:

        folder = (
            SPLIT_DIR /
            split_name /
            class_name
        )

        count = len([
            f for f in folder.iterdir()
            if f.is_file()
        ])

        print(
            f"{class_name:25} {count}"
        )

In [ ]:
def get_hashes(directory):

    hashes = {}

    for file in Path(directory).rglob("*"):

        if file.is_file():

            try:
                hashes[calculate_md5(file)] = str(file)
            except:
                pass

    return hashes


train_hashes = get_hashes(SPLIT_DIR / "train")
val_hashes = get_hashes(SPLIT_DIR / "val")
test_hashes = get_hashes(SPLIT_DIR / "test")

train_val = set(train_hashes) & set(val_hashes)
train_test = set(train_hashes) & set(test_hashes)
val_test = set(val_hashes) & set(test_hashes)

print("Train ∩ Validation:", len(train_val))
print("Train ∩ Test:", len(train_test))
print("Validation ∩ Test:", len(val_test))

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

In [ ]:
from torchvision import transforms

IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(
        degrees=10
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
from torchvision.datasets import ImageFolder

train_dataset = ImageFolder(
    root=str(SPLIT_DIR / "train"),
    transform=train_transform
)

val_dataset = ImageFolder(
    root=str(SPLIT_DIR / "val"),
    transform=val_test_transform
)

test_dataset = ImageFolder(
    root=str(SPLIT_DIR / "test"),
    transform=val_test_transform
)

print("Classes:")
print(train_dataset.classes)

print("\nClass mapping:")
print(train_dataset.class_to_idx)

print("\nDataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders ready.")

In [ ]:
images, labels = next(iter(train_loader))

print("Image tensor shape:")
print(images.shape)

print("\nLabels shape:")
print(labels.shape)

print("\nLabels:")
print(labels[:10])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 8))

for i in range(9):

    image = images[i].permute(1, 2, 0).numpy()

    image = image * std + mean
    image = np.clip(image, 0, 1)

    plt.subplot(3, 3, i + 1)

    plt.imshow(image)

    plt.title(
        train_dataset.classes[labels[i]]
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import json

dataset_stats = {
    "classes": classes,
    "total_original_images": len(df),
    "valid_images": int(df["valid"].sum()),
    "invalid_images": int((~df["valid"]).sum()),
    "train_images": len(train_dataset),
    "validation_images": len(val_dataset),
    "test_images": len(test_dataset),
    "image_size": IMAGE_SIZE,
    "split": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15
    },
    "random_seed": 42
}

with open(
    REPORT_DIR / "dataset_statistics.json",
    "w"
) as f:

    json.dump(
        dataset_stats,
        f,
        indent=4
    )

print("Dataset statistics saved.")

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
from torch import nn


class AgroVisionCNN(nn.Module):

    def __init__(self, num_classes=5):

        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d((1, 1)),

            nn.Flatten(),

            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x

In [ ]:
model = AgroVisionCNN(
    num_classes=5
).to(device)

print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
def evaluate(model, loader, criterion):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [ ]:
from pathlib import Path

import torch
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

# Ensure SPLIT_DIR and device are defined
SPLIT_DIR = Path("/content/dataset_b/split")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dynamic fallback to instantiate loaders if they are missing from memory
if 'train_loader' not in globals() or 'val_loader' not in globals():
    print("Re-initializing datasets and DataLoaders...")
    IMAGE_SIZE = 224

    train_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_test_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = ImageFolder(root=str(SPLIT_DIR / "train"), transform=train_transform)
    val_dataset = ImageFolder(root=str(SPLIT_DIR / "val"), transform=val_test_transform)

    BATCH_SIZE = 32
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

EPOCHS = 20

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    val_loss, val_acc = evaluate(
        model,
        val_loader,
        criterion
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            model.state_dict(),
            "/content/best_agrovision_cnn.pth"
        )

print("\nBest validation accuracy:", best_val_acc)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(
    history["train_acc"],
    label="Training Accuracy"
)

plt.plot(
    history["val_acc"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("AgroVision CNN Accuracy")

plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history["train_loss"],
    label="Training Loss"
)

plt.plot(
    history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("AgroVision CNN Loss")

plt.legend()
plt.show()

In [ ]:
model.load_state_dict(
    torch.load(
        "/content/best_agrovision_cnn.pth",
        map_location=device
    )
)

model.eval()

print("Best model loaded.")

In [ ]:
test_loss, test_accuracy = evaluate(
    model,
    test_loader,
    criterion
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

In [ ]:
class_names = test_dataset.classes

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4
    )
)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(
    all_labels,
    all_predictions
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(8, 8))

disp.plot(
    ax=ax,
    xticks_rotation=45
)

plt.title("AgroVision CNN Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_names": class_names,
        "image_size": 224
    },
    "/content/AgroVision_CNN_baseline.pth"
)

print("CNN model saved.")

In [ ]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load best saved model
model.load_state_dict(
    torch.load(
        "/content/best_agrovision_cnn.pth",
        map_location=device
    )
)

model.to(device)
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

In [ ]:
test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
class_names = test_dataset.classes

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4
    )
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(
    all_labels,
    all_predictions
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, xticks_rotation=45)
plt.title("AgroVision CNN - Test Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

report = classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()

report_df.to_csv(
    "/content/AgroVision_CNN_results.csv"
)

print("CNN results saved.")

In [ ]:
import shutil

SOURCE_DIR = "/content/dataset_b/split"
OUTPUT_ZIP = "/content/AgroVision_DatasetB_Cleaned_Split"

shutil.make_archive(
    OUTPUT_ZIP,
    "zip",
    SOURCE_DIR
)

print("ZIP created successfully:")
print(OUTPUT_ZIP + ".zip")

In [ ]:
from google.colab import files

files.download(
    "/content/AgroVision_DatasetB_Cleaned_Split.zip"
)

In [ ]:
# CNN OUTPUT CHECK

model.eval()

with torch.no_grad():

    batch = next(iter(test_loader))

    print("Type of batch:", type(batch))

In [ ]:
import os

model_file_path = "/content/AgroVision_CNN_baseline.pth"

if os.path.exists(model_file_path):
    print(f"The model file exists at: {model_file_path}")
else:
    print(f"The model file does NOT exist at: {model_file_path}")